In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw/ids")

list(DATA_DIR.glob("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"))

[PosixPath('../data/raw/ids/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')]

In [2]:
csv_files = list(DATA_DIR.glob("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"))

df = pd.read_csv(csv_files[0])

print("File:", csv_files[0].name)
print("Shape:", df.shape)

df.head()

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Shape: (225745, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
print("Number of columns:", len(df.columns))

for i, column in enumerate(df.columns):
    print(i, repr(column))

Number of columns: 79
0 ' Destination Port'
1 ' Flow Duration'
2 ' Total Fwd Packets'
3 ' Total Backward Packets'
4 'Total Length of Fwd Packets'
5 ' Total Length of Bwd Packets'
6 ' Fwd Packet Length Max'
7 ' Fwd Packet Length Min'
8 ' Fwd Packet Length Mean'
9 ' Fwd Packet Length Std'
10 'Bwd Packet Length Max'
11 ' Bwd Packet Length Min'
12 ' Bwd Packet Length Mean'
13 ' Bwd Packet Length Std'
14 'Flow Bytes/s'
15 ' Flow Packets/s'
16 ' Flow IAT Mean'
17 ' Flow IAT Std'
18 ' Flow IAT Max'
19 ' Flow IAT Min'
20 'Fwd IAT Total'
21 ' Fwd IAT Mean'
22 ' Fwd IAT Std'
23 ' Fwd IAT Max'
24 ' Fwd IAT Min'
25 'Bwd IAT Total'
26 ' Bwd IAT Mean'
27 ' Bwd IAT Std'
28 ' Bwd IAT Max'
29 ' Bwd IAT Min'
30 'Fwd PSH Flags'
31 ' Bwd PSH Flags'
32 ' Fwd URG Flags'
33 ' Bwd URG Flags'
34 ' Fwd Header Length'
35 ' Bwd Header Length'
36 'Fwd Packets/s'
37 ' Bwd Packets/s'
38 ' Min Packet Length'
39 ' Max Packet Length'
40 ' Packet Length Mean'
41 ' Packet Length Std'
42 ' Packet Length Variance'
43 'FIN 

In [11]:
print(df[" Label"].value_counts(dropna=False))

 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [12]:
print(df[" Label"].value_counts(normalize=True, dropna=False) * 100)

 Label
DDoS      56.713105
BENIGN    43.286895
Name: proportion, dtype: float64


In [13]:
# Clean whitespace from column names
df.columns = df.columns.str.strip()

print(df.columns[-5:])

Index(['Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label'], dtype='object')


In [14]:
print(df["Label"].value_counts())

Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [15]:
df["target"] = df["Label"].map({
    "BENIGN": 0,
    "DDoS": 1
})

print(df["target"].value_counts())

target
1    128027
0     97718
Name: count, dtype: int64


In [16]:
print("Unmapped labels:")
print(df.loc[df["target"].isna(), "Label"].unique())

Unmapped labels:
[]


In [17]:
selected_columns = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Average Packet Size",
    "Flow Packets/s",
    "Flow Bytes/s",
    "Label",
    "target"
]

df[selected_columns].head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Average Packet Size,Flow Packets/s,Flow Bytes/s,Label,target
0,3,2,0,12,0,9.0,666666.66670,4.000000e+06,BENIGN,0
1,109,1,1,6,6,9.0,18348.62385,1.100917e+05,BENIGN,0
2,52,1,1,6,6,9.0,38461.53846,2.307692e+05,BENIGN,0
3,34,1,1,6,6,9.0,58823.52941,3.529412e+05,BENIGN,0
4,3,2,0,12,0,9.0,666666.66670,4.000000e+06,BENIGN,0


In [18]:
print(df["Label"].value_counts())
print(df["target"].value_counts())
print(df.loc[df["target"].isna(), "Label"].unique())

Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64
target
1    128027
0     97718
Name: count, dtype: int64
[]


In [23]:
from app.ml.dataset_adapter import adapt_cic_ids2017

model_df = adapt_cic_ids2017(df)

print("Original shape:", df.shape)
print("Adapted shape:", model_df.shape)

model_df.head()

Original shape: (225745, 80)
Adapted shape: (225745, 12)


,duration,forward_packets,backward_packets,total_packets,forward_bytes,backward_bytes,total_bytes,average_packet_size,packets_per_second,bytes_per_second,protocol,target
0,0.000003,2,0,2,12,0,12,9.0,666666.66670,4.000000e+06,OTHER,0
1,0.000109,1,1,2,6,6,12,9.0,18348.62385,1.100917e+05,OTHER,0
2,0.000052,1,1,2,6,6,12,9.0,38461.53846,2.307692e+05,OTHER,0
3,0.000034,1,1,2,6,6,12,9.0,58823.52941,3.529412e+05,OTHER,0
4,0.000003,2,0,2,12,0,12,9.0,666666.66670,4.000000e+06,OTHER,0


In [22]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/upendrakumaryadav415/Desktop/AI Project/ai-network-traffic-analysis


In [24]:
print(model_df.columns.tolist())

['duration', 'forward_packets', 'backward_packets', 'total_packets', 'forward_bytes', 'backward_bytes', 'total_bytes', 'average_packet_size', 'packets_per_second', 'bytes_per_second', 'protocol', 'target']


In [25]:
print(model_df.columns.tolist())
print(model_df["target"].value_counts(dropna=False))
print("Unmapped targets:", model_df["target"].isna().sum())

['duration', 'forward_packets', 'backward_packets', 'total_packets', 'forward_bytes', 'backward_bytes', 'total_bytes', 'average_packet_size', 'packets_per_second', 'bytes_per_second', 'protocol', 'target']
target
1    128027
0     97718
Name: count, dtype: int64
Unmapped targets: 0


In [26]:
print(model_df.columns.tolist())

['duration', 'forward_packets', 'backward_packets', 'total_packets', 'forward_bytes', 'backward_bytes', 'total_bytes', 'average_packet_size', 'packets_per_second', 'bytes_per_second', 'protocol', 'target']


In [27]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("Project root:", Path.cwd().parent)
print("app exists:", (Path.cwd().parent / "app").exists())
print("adapter exists:", (Path.cwd().parent / "app/ml/dataset_adapter.py").exists())

Current directory: /Users/upendrakumaryadav415/Desktop/AI Project/ai-network-traffic-analysis/notebooks
Project root: /Users/upendrakumaryadav415/Desktop/AI Project/ai-network-traffic-analysis
app exists: True
adapter exists: True


In [28]:
numeric_features = [
    "duration",
    "forward_packets",
    "backward_packets",
    "total_packets",
    "forward_bytes",
    "backward_bytes",
    "total_bytes",
    "average_packet_size",
    "packets_per_second",
    "bytes_per_second",
]

print(model_df[numeric_features].dtypes)

duration               float64
forward_packets          int64
backward_packets         int64
total_packets            int64
forward_bytes            int64
backward_bytes           int64
total_bytes              int64
average_packet_size    float64
packets_per_second     float64
bytes_per_second       float64
dtype: object


In [29]:
print("NaN values:")
print(model_df[numeric_features].isna().sum().sum())

print("\nInfinite values:")
print(
    np.isinf(
        model_df[numeric_features].to_numpy()
    ).sum()
)

NaN values:
0

Infinite values:
0


In [30]:
model_df[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
duration,225745.0,16.241649,3.152437e+01,-1.000000e-06,0.071180,1.452333,8.805237,1.199999e+02
forward_packets,225745.0,4.874916,1.542287e+01,1.000000e+00,2.000000,3.000000,5.000000,1.932000e+03
backward_packets,225745.0,4.572775,2.175536e+01,0.000000e+00,1.000000,4.000000,5.000000,2.942000e+03
total_packets,225745.0,9.447691,3.678550e+01,2.000000e+00,4.000000,7.000000,9.000000,4.813000e+03
forward_bytes,225745.0,939.463346,3.249403e+03,0.000000e+00,26.000000,30.000000,63.000000,1.830120e+05
backward_bytes,225745.0,5960.477455,3.921834e+04,0.000000e+00,0.000000,164.000000,11601.000000,5.172346e+06
total_bytes,225745.0,6899.940800,3.957046e+04,0.000000e+00,30.000000,826.000000,11633.000000,5.196726e+06
average_packet_size,225745.0,574.568843,6.260962e+02,0.000000e+00,7.500000,141.000000,1291.888889,2.528000e+03
packets_per_second,225745.0,14239.056315,1.150957e+05,-2.000000e+06,0.618293,5.170831,70.302476,3.000000e+06
bytes_per_second,225745.0,585305.709835,1.688425e+07,-1.200000e+07,12.066407,1130.278240,21565.555920,2.070000e+09


In [31]:
print(
    "Zero-duration flows:",
    (model_df["duration"] == 0).sum()
)

print(
    "Negative-duration flows:",
    (model_df["duration"] < 0).sum()
)

Zero-duration flows: 34
Negative-duration flows: 2


In [32]:
packet_check = (
    model_df["forward_packets"]
    + model_df["backward_packets"]
    == model_df["total_packets"]
)

byte_check = (
    model_df["forward_bytes"]
    + model_df["backward_bytes"]
    == model_df["total_bytes"]
)

print("Packet totals correct:", packet_check.all())
print("Byte totals correct:", byte_check.all())

Packet totals correct: True
Byte totals correct: True


In [33]:
processed_path = Path("../data/processed/ids/ddos_binary.csv")

model_df.to_csv(
    processed_path,
    index=False
)

print("Saved:", processed_path)
print("Shape:", model_df.shape)

Saved: ../data/processed/ids/ddos_binary.csv
Shape: (225745, 12)


In [34]:
print("NaN:", model_df[numeric_features].isna().sum().sum())

print(
    "Infinite:",
    np.isinf(model_df[numeric_features].to_numpy()).sum()
)

print(
    "Zero duration:",
    (model_df["duration"] == 0).sum()
)

print(
    "Negative duration:",
    (model_df["duration"] < 0).sum()
)

print(
    "Packet totals correct:",
    packet_check.all()
)

print(
    "Byte totals correct:",
    byte_check.all()
)

NaN: 0
Infinite: 0
Zero duration: 34
Negative duration: 2
Packet totals correct: True
Byte totals correct: True


In [35]:
before = len(model_df)

model_df = model_df[
    model_df["duration"] >= 0
].copy()

after = len(model_df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 225745
Rows after: 225743
Rows removed: 2


In [36]:
print("Negative duration:",
      (model_df["duration"] < 0).sum())

print("NaN:",
      model_df[numeric_features].isna().sum().sum())

print("Infinite:",
      np.isinf(
          model_df[numeric_features].to_numpy()
      ).sum())

Negative duration: 0
NaN: 0
Infinite: 0


In [37]:
processed_path = Path("../data/processed/ids/ddos_binary.csv")

model_df.to_csv(
    processed_path,
    index=False
)

print("Saved:", processed_path)
print("Final shape:", model_df.shape)

Saved: ../data/processed/ids/ddos_binary.csv
Final shape: (225743, 12)


In [38]:
print(model_df.shape)
print(model_df["target"].value_counts())

(225743, 12)
target
1    128027
0     97716
Name: count, dtype: int64


In [39]:
model_df = model_df[
    model_df["duration"] >= 0
].copy()

print("Final shape:", model_df.shape)
print("Negative duration:",
      (model_df["duration"] < 0).sum())

Final shape: (225743, 12)
Negative duration: 0


In [40]:
processed_path = Path("../data/processed/ids/ddos_binary.csv")

model_df.to_csv(
    processed_path,
    index=False
)

print("Saved:", processed_path)

Saved: ../data/processed/ids/ddos_binary.csv


In [41]:
#Train

In [42]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["target"])
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training features: (180594, 11)
Testing features: (45149, 11)

Training labels:
target
1    102421
0     78173
Name: count, dtype: int64

Testing labels:
target
1    25606
0    19543
Name: count, dtype: int64


In [43]:
from app.ml.preprocess import prepare_features

X_train = prepare_features(X_train)
X_test = prepare_features(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print(X_train.head())

X_train: (180594, 11)
X_test: (45149, 11)
         duration  forward_packets  backward_packets  total_packets  \
183794   0.037619                3                 4              7   
5348     0.000003                2                 0              2   
75617   80.285313                8                 5             13   
108921   0.035254                3                 6              9   
125564  91.502032                8                 6             14   

        forward_bytes  backward_bytes  total_bytes  average_packet_size  \
183794             26           11601        11627          1661.000000   
5348               37               0           37            34.000000   
75617              56           11607        11663           897.615385   
108921             26           11601        11627          1291.888889   
125564             56           11601        11657           833.071429   

        packets_per_second  bytes_per_second protocol  
183794          186.0761

In [44]:
protocol = "OTHER"

In [45]:
X_train = X_train.drop(columns=["protocol"])
X_test = X_test.drop(columns=["protocol"])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (180594, 10)
Testing shape: (45149, 10)


In [46]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [47]:
y_pred = model.predict(X_test)

print("Predictions generated:", len(y_pred))

Predictions generated: 45149


In [48]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["BENIGN", "DDoS"]
    )
)

Accuracy : 0.9993798312254978
Precision: 0.999531286618233
Recall   : 0.9993751464500508
F1 Score : 0.9994532104358694

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     19543
        DDoS       1.00      1.00      1.00     25606

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [49]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[19531    12]
 [   16 25590]]
